# SupeRR Selector — player ratings

This notebook reads the silver files and builds two lists: batters and bowlers.

Each player has a baseline from their usual rates, then three pressure bands, a within-match comparison, a clutch term, form, and consistency. A career rate is the decay-weighted mean of the rows that player has. A season with no rows is left out of both the total and the weight. The weights are 0.6561, 0.729, 0.81, 0.9, and 1 for 2022 through 2026.

The calculations live in `player_ratings.py`. This notebook runs them and shows the tables. The silver files are not changed.

## Load and tag balls

A legal ball is not a wide, a no-ball, or a super over. A wide or a no-ball adds one conceded run. Bye and leg-bye amounts are not in the file, and neither are extra runs beyond that one-run penalty, so a team total is still a lower bound. An ordinary chase target is the first-innings total plus one. A DLS target is used as supplied.

A legal ball is not a wide, a no-ball, or a super over. Pressure is the occasion times the competition times the ball, plus a shock from the last six balls and a break from the previous eighteen. The current ball is not included. The shock starts with dots and wickets for a batter, and fours and sixes for a bowler, then adds the wickets above xW and the charged runs above xR on those same six balls. Bowling bands are cut inside pace and inside spin, so the new ball does not put more pace bowlers into the high band. A recognized batter is anyone at positions 1 to 7. Zero, medium, and high are the bottom half, the next 30 percent, and the top fifth of that pressure.

In [1]:
from pathlib import Path

import pandas as pd

import player_ratings as ratings

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 30)

balls, matches, comps = ratings.load()
work, legal = ratings.add_situation(balls)
print("deliveries", len(balls), "legal, excluding super overs", len(legal))
print("five-season weight total", round(ratings.W_SUM, 4))
bands = ratings.band_summary(legal)
display(bands.assign(share=lambda frame: (100 * frame["share"]).round(1)))

deliveries 200999 legal, excluding super overs 192675
five-season weight total 4.0951


,balls,share
band,,
zero,94837,49.2
medium,59221,30.7
high,38617,20.0


## Elo

Team strength updates after each match. Batting strength and bowling strength update after each legal ball, on separate IPL and international ladders. A player with few balls is held near 1500, and that hold loosens as the career lengthens. Venue strength uses only balls already bowled at that ground. Expected runs and the expected wicket chance are also taken from balls already bowled, split by phase, pace or spin, and ladder, and pulled toward the coarser rate with 30 balls. Before any ball, the starting assumption is 1.0 run and a 0.05 wicket chance. A batter is priced on bat runs. A bowler is priced on charged runs, which are 0 on a bye, leg-bye, or penalty, and the bowling rating uses that same charged total.

In [2]:
charged_gap = legal["bat_runs"] - legal["charged_runs"]
print("balls where bat runs differ from charged runs", int(charged_gap.ne(0).sum()))
print("sum of bat runs minus charged runs", round(float(charged_gap.sum()), 3))
legal = ratings.fit_elo(legal)
summary = legal.attrs["elo_summary"]
for key, value in summary.items():
    if isinstance(value, str):
        print(key + ":", value)
    else:
        print(key + ":", round(value, 1))

balls where bat runs differ from charged runs 1
sum of bat runs minus charged runs 1.0


k_run: 4.0
k_wicket: 16.0
k_team: 20.0
batter_elo_min: 932.8
batter_elo_max: 2587.3
batter_elo_sd: 148.0
bowler_elo_min: 1061.0
bowler_elo_max: 2109.0
bowler_elo_sd: 121.0
team_elo_min: 1361.3
team_elo_max: 1647.0
grounds: 97
step_sizes: as written


## Batter scores

Baseline is strike rate and average against the phases that batter actually plays. The three bands are the strength-adjusted ball value inside zero, medium, and high pressure. Peers compare an innings with teammates and the opposition, and the opposition gap is capped at 3, the same cap as the teammate gap. Clutch uses a high-band spell against the player's own rate. A spell below that rate stays negative. A spell above it is scaled by how far the player also beat their teammates, and a negative teammate gap does not add a bonus. Form uses up to eight earlier innings and is pulled toward the middle when that sample is short. Consistency prefers a narrower spread of match values.

Each term is a z-score inside the batting role, so a top-order batter is compared with other top-order batters, a middle-order batter with the middle order, and a lower-order batter with the lower order. Inside each role the reference is still players with at least 30 decay-weighted legal balls, and each z-score stops at 3. The composite is 0.20 baseline, 0.10 zero, 0.15 medium, 0.20 high, 0.15 clutch, 0.10 peers, 0.05 form, and 0.05 consistency. The extra 0.05, which the eight original shares did not account for, is on the baseline.

In [3]:
batters = ratings.score_batters(legal)
print("batters", len(batters), "ranked", int(batters["ranked"].sum()))
batter_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "runs", "strike_rate", "average", "n_eff", "high_n_eff",
]
display(ratings.top_table(batters, batter_cols))
print("roles in the batter top 10")
print(batters.loc[batters["rank"].le(10), "role"].value_counts().to_string())

batters 888 ranked 223


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,runs,strike_rate,average,n_eff,high_n_eff
0,1.0,Maaz Sadaqat,top order,1.730,2.398,2.282,0.988,2.406,0.007,2.858,1.272,0.837,404,177.193,80.800,213.500,37.700
1,2.0,Rilee Rossouw,top order,1.310,1.384,2.014,0.229,2.998,0.007,1.955,-0.611,0.640,717,172.771,32.591,302.519,56.303
2,3.0,Tim David,middle order,1.255,2.379,1.692,1.942,0.462,-0.427,3.000,0.544,-0.732,2153,177.202,34.726,1002.709,314.757
3,4.0,Mahendra Singh Dhoni,lower order,1.174,1.704,-0.937,1.378,1.051,3.000,1.597,0.044,-2.033,690,150.000,31.364,352.416,128.987
4,5.0,George Linde,lower order,1.170,1.050,-0.442,2.004,1.957,0.314,1.731,2.210,-0.376,227,162.143,18.917,130.740,61.550
5,6.0,Vaibhav Sooryavanshi,top order,1.078,2.506,3.000,3.000,0.117,-2.255,3.000,-0.160,-3.000,1509,223.887,43.114,652.100,45.700
6,7.0,Tom Banton,middle order,1.015,1.214,1.576,0.689,2.280,-0.061,1.396,-0.747,-0.753,437,159.489,31.214,239.776,68.642
7,8.0,Donovan Ferreira,middle order,1.008,0.960,0.466,2.199,0.976,2.066,0.990,-0.944,-2.352,561,164.035,24.391,317.418,85.793
8,9.0,Dinesh Karthik,middle order,1.004,1.528,1.128,1.870,-0.487,2.116,1.792,0.111,-1.996,1061,166.562,31.206,452.142,152.280
9,10.0,Patrick Cummins,lower order,0.970,1.084,2.178,0.548,0.097,2.919,1.408,-0.053,-2.848,389,158.130,20.474,199.933,92.577


roles in the batter top 10
role
middle order    4
top order       3
lower order     3


## Bowler scores

The same shape. Baseline uses economy and balls per wicket, against the usual rate for that bowler's pace or spin and the phases they bowl. A lower rate is a higher score. The high band, clutch, and peers use the same direction.

In [4]:
bowlers = ratings.score_bowlers(legal, work)
print("bowlers", len(bowlers), "ranked", int(bowlers["ranked"].sum()))
bowler_cols = [
    "rank", "player_name", "role", "composite", "baseline",
    "band_zero", "band_medium", "band_high", "clutch", "peers", "form", "consistency",
    "wickets", "economy", "balls_per_wicket", "n_eff", "high_n_eff",
]
display(ratings.top_table(bowlers, bowler_cols))
print("roles in the bowler top 10")
print(bowlers.loc[bowlers["rank"].le(10), "role"].value_counts().to_string())

bowlers 647 ranked 232


,rank,player_name,role,composite,baseline,band_zero,band_medium,band_high,clutch,peers,form,consistency,wickets,economy,balls_per_wicket,n_eff,high_n_eff
0,1.0,Corbin Bosch,Pace,1.305,1.099,1.407,-0.274,2.333,3.000,0.498,0.310,0.073,33.0,8.332,11.152,346.900,85.700
1,2.0,Matthew Humphreys,Spin,1.239,1.189,1.196,0.201,1.400,2.575,1.474,0.077,0.688,20.0,6.866,15.250,277.028,31.330
2,3.0,Imad Wasim,Spin,1.042,1.437,1.117,0.394,1.418,0.017,2.912,0.620,-0.497,15.0,5.951,16.400,188.568,39.933
3,4.0,Mujeeb Ur Rahman,Spin,0.926,1.272,-0.543,0.566,1.202,1.132,1.758,0.504,0.594,48.0,6.783,17.083,675.993,117.428
4,5.0,Jasprit Bumrah,Pace,0.849,1.256,0.488,1.492,0.305,-0.419,3.000,0.601,-0.056,85.0,6.920,20.106,1455.083,302.538
5,6.0,Dwayne Bravo,Pace,0.839,0.991,-0.015,2.204,1.016,0.740,-0.065,-1.317,1.391,16.0,7.923,13.062,137.125,38.710
6,7.0,Sunil Narine,Spin,0.814,0.441,1.198,-0.624,1.004,0.844,3.000,0.180,1.255,64.0,6.773,23.906,1246.702,252.413
7,8.0,Shoriful Islam,Pace,0.812,0.539,-0.895,1.022,1.675,2.337,-0.278,-0.935,0.594,41.0,7.972,19.293,650.221,86.999
8,9.0,Haider Ali,Spin,0.799,1.170,0.043,0.810,1.845,-0.411,1.839,-1.924,0.892,15.0,6.333,18.000,247.800,58.500
9,10.0,Salman Mirza,Pace,0.767,1.568,1.201,0.653,0.688,-0.309,0.895,1.066,0.024,21.0,6.640,15.619,303.000,41.000


roles in the bowler top 10
role
Pace    5
Spin    5


## Rankings

A player is ranked in a skill with at least 120 decay-weighted legal balls, and at least 30 of those in the high band. Anyone under the line stays in the full table with no rank. The two lists are saved under `ratings/`.

In [5]:
out = Path("ratings")
out.mkdir(exist_ok=True)
batters.to_csv(out / "batters.csv", index=False)
bowlers.to_csv(out / "bowlers.csv", index=False)
print("wrote", out / "batters.csv", "and", out / "bowlers.csv")
print("decision: floor is 120 decay-weighted legal balls and 30 high-band balls")
print("decision: career rate is the decay-weighted mean of rows played; the five weights sum to", round(ratings.W_SUM, 4))
print("decision: elo step sizes", summary["step_sizes"])

wrote ratings\batters.csv and ratings\bowlers.csv
decision: floor is 120 decay-weighted legal balls and 30 high-band balls
decision: career rate is the decay-weighted mean of rows played; the five weights sum to 4.0951
decision: elo step sizes as written
